# Sentence Memorability — Phase 2: Advanced Inferential Analysis

This notebook implements four blocks of advanced inferential analysis for the Sentence Memorability project:

| Block | Method | Purpose |
|-------|--------|---------|
| 1 | Repeated-Measures ANOVA | Test within-participant condition effects and interactions |
| 2 | Linear Mixed Models (LMMs) | Trial-level modelling with participant random intercepts |
| 3 | OLS & Logistic Regression | Sentence-level and trial-level prediction of memorability |
| 4 | Signal Detection (d′) | Separate true sensitivity from response bias |

**Dependencies:** pingouin, statsmodels, scipy, pandas, numpy, matplotlib, seaborn


## Setup and Imports

Import all required libraries. Warnings are suppressed for clean output. A fixed random seed ensures reproducibility of any stochastic routines.


In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
from io import StringIO
from scipy import stats
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

warnings.filterwarnings('ignore')
np.random.seed(42)


## Output Paths and Logging Utilities

- `log()` prints to console and writes to an in-memory buffer for later saving.
- `save_fig()` saves matplotlib figures at 200 DPI and closes them to free memory.
- `save_results()` flushes the full log buffer to `analysis_results_phase2.txt`.


In [ ]:
OUT_DIR = os.getcwd()
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(OUT_DIR, 'analysis_results_phase2.txt')
log_buf = StringIO()

def log(msg=''):
    """Log to console and the in-memory buffer."""
    print(msg)
    log_buf.write(msg + '\n')

def save_results():
    """Write the buffered log to disk."""
    with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
        f.write(log_buf.getvalue())
    log(f'\n[Saved results to {RESULTS_FILE}]')

def save_fig(fig, name):
    """Save and close a matplotlib figure."""
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    log(f'  -> Saved figures/{name}')


## Data Loading

Load the pre-processed Phase 1 outputs by dynamically importing `code.py` and calling its `phase1_load()` function. This returns:
- **targets_repeat** — Trial-level repeated-target rows for IR analyses (5,264 rows)
- **hits** — Hit trials with RT and WR fields
- **false_alarms / fa_rate_df** — False-alarm counts and participant-level FA rates
- **per_sentence_ir / per_sentence_wr** — Sentence-level memorability scores for regression

All cleaning, validation filtering, block assignment, and stimulus parsing was done in Phase 1. Phase 2 consumes the already-structured tables.


In [ ]:
def load_data():
    """Import and run phase1_load from the existing code.py.

    This function dynamically loads code.py and calls its phase1_load()
    entry point, which parses all .log files, applies validation filtering,
    assigns blocks, and returns structured dataframes.
    """
    import importlib.util
    code_path = os.path.join(OUT_DIR, 'code.py')
    spec = importlib.util.spec_from_file_location("code_phase1", code_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.phase1_load()

(df_valid, val_results, targets_repeat, hits,
 false_alarms, per_sentence_ir, per_sentence_wr, fa_rate_df) = load_data()

print(f'Data loaded: {len(targets_repeat)} trials, '
      f'{targets_repeat["participant_ID"].nunique()} participants, '
      f'{len(per_sentence_ir)} sentences')


---
## Block 1: Repeated-Measures ANOVA

### What is RM-ANOVA?
Repeated-Measures ANOVA tests whether mean outcomes differ across conditions when the **same participants** are measured in every condition. It partitions variance into condition effects (signal) and residual error (noise).

### Why use it here?
The design is fully within-participant (every participant saw all 4 sentence types in both voices). RM-ANOVA leverages this pairing to reduce noise from individual baseline differences.

### Design decomposition
Because `pingouin` supports at most 2 within-factors, the full 2×2×2 design is decomposed into **three separate 2×2 RM-ANOVAs per DV** (SubjectMem × ObjectMem, SubjectMem × Voice, ObjectMem × Voice), plus a **one-way RM-ANOVA for Block** effects.

**DVs analysed:** IR Hit Rate, WR Accuracy, Reaction Time (RT_IR)


### Helper Function: `_run_two_way_rmanova`

This function runs a single 2×2 RM-ANOVA for a given pair of within-participant factors:
1. **Aggregates** trial-level data to participant × factor1 × factor2 cell means
2. **Filters** to participants with all 4 cells (complete cases only)
3. **Runs** the RM-ANOVA via `pingouin.rm_anova()` with partial eta-squared effect size
4. **Post-hoc:** If the interaction is significant (p < .05), runs simple-effects paired t-tests at each level of factor1


In [ ]:
def _run_two_way_rmanova(pg, data, dv, subject, factor1, factor2, label):
    """Helper: run a 2×2 RM-ANOVA and log the results."""
    # Aggregate to participant × factor1 × factor2 means
    agg = data.groupby([subject, factor1, factor2])[dv].mean().reset_index()
    counts = agg.groupby(subject).size()
    valid = counts[counts == 4].index
    agg_valid = agg[agg[subject].isin(valid)].copy()
    log(f"\n  {label}: {factor1} × {factor2} (N={len(valid)} participants)")
    if len(valid) >= 10:
        aov = pg.rm_anova(
            data=agg_valid, dv=dv, subject=subject,
            within=[factor1, factor2], effsize='np2'
        )
        log(aov.to_string(index=False))

        # Post-hoc if interaction is significant
        int_src = f'{factor1} * {factor2}'
        int_row = aov[aov['Source'] == int_src]
        if len(int_row) > 0 and int_row.iloc[0].get('p-unc', 1) < 0.05:
            log(f"\n  {int_src} interaction is significant — simple effects:")
            for level in agg_valid[factor1].unique():
                sub = agg_valid[agg_valid[factor1] == level]
                sagg = sub.groupby([subject, factor2])[dv].mean().reset_index()
                wide = sagg.pivot(index=subject, columns=factor2, values=dv).dropna()
                cols = sorted(wide.columns)
                if len(cols) == 2:
                    t_stat, p_val = stats.ttest_rel(wide[cols[0]], wide[cols[1]])
                    log(f"    {factor1}={level}: {factor2} {cols[0]} vs {cols[1]}: "
                        f"t={t_stat:.4f}, p={p_val:.4e}")
    else:
        log("  Not enough participants with all cells. Skipping.")



### Main Function: `block1_rmanova`

This function orchestrates all RM-ANOVAs for Phase 2 Block 1:

1. **Prepares cell-level data** — Aggregates trial data to participant × SubjectMem × ObjectMem × Voice means for each DV (IR accuracy, WR accuracy, median RT)
2. **Runs nine 2×2 RM-ANOVAs** — Three factor pairs (SubjectMem×ObjectMem, SubjectMem×Voice, ObjectMem×Voice) × three DVs
3. **Runs one-way Block ANOVA** — Tests whether hit rate declines across experimental blocks (fatigue effect)
4. **Generates interaction plots** — Two multi-panel figures showing SubjectMem×ObjectMem and Voice×SubjectMem interaction patterns

The function also runs Bonferroni-corrected post-hoc pairwise comparisons for the Block effect.


In [ ]:
def block1_rmanova(targets_repeat, hits):
    log("\n" + "=" * 70)
    log("BLOCK 1: REPEATED-MEASURES ANOVA")
    log("=" * 70)
    log("  Note: pingouin supports max 2 within-factors, so the 2×2×2 design")
    log("  is decomposed into three separate 2×2 RM-ANOVAs per DV.")

    try:
        import pingouin as pg
    except ImportError:
        log("  ERROR: pingouin not installed. Run: pip install pingouin")
        return

    # Only WR trials with usable accuracy
    wr_hits = hits.dropna(subset=['Accuracy_WR'])

    # Prepare cell-level data for each DV
    cell_ir = targets_repeat.groupby(
        ['participant_ID', 'SubjectMem', 'ObjectMem', 'Voice']
    )['Accuracy_IR'].mean().reset_index()
    cell_ir.columns = ['participant_ID', 'SubjectMem', 'ObjectMem', 'Voice', 'HitRate']


    # WR accuracy: restrict to participants with all 8 cells
    cell_wr_valid = pd.DataFrame()
    valid_pids_wr = []
    if len(wr_hits) > 0:
        cell_wr = wr_hits.groupby(
            ['participant_ID', 'SubjectMem', 'ObjectMem', 'Voice']
        )['Accuracy_WR'].mean().reset_index()
        cell_wr.columns = ['participant_ID', 'SubjectMem', 'ObjectMem', 'Voice', 'WR_Rate']
        cell_counts_wr = cell_wr.groupby('participant_ID').size()
        valid_pids_wr = cell_counts_wr[cell_counts_wr == 8].index
        cell_wr_valid = cell_wr[cell_wr['participant_ID'].isin(valid_pids_wr)].copy()

    # RT: median per cell, require full 8-cell coverage
    cell_rt = hits.dropna(subset=['RT_IR']).groupby(
        ['participant_ID', 'SubjectMem', 'ObjectMem', 'Voice']
    )['RT_IR'].median().reset_index()
    cell_rt.columns = ['participant_ID', 'SubjectMem', 'ObjectMem', 'Voice', 'MedianRT']
    cell_counts_rt = cell_rt.groupby('participant_ID').size()
    valid_pids_rt = cell_counts_rt[cell_counts_rt == 8].index
    cell_rt_valid = cell_rt[cell_rt['participant_ID'].isin(valid_pids_rt)].copy()

    # ────────────────────────────────────────────────────────────
    # 1A. IR Accuracy — three 2×2 RM-ANOVAs
    # ────────────────────────────────────────────────────────────
    log("\n── 1A. RM-ANOVAs: IR Accuracy (Hit Rate) ──")
    _run_two_way_rmanova(pg, cell_ir, 'HitRate', 'participant_ID',
                         'SubjectMem', 'ObjectMem', 'IR Accuracy')
    _run_two_way_rmanova(pg, cell_ir, 'HitRate', 'participant_ID',
                         'SubjectMem', 'Voice', 'IR Accuracy')
    _run_two_way_rmanova(pg, cell_ir, 'HitRate', 'participant_ID',
                         'ObjectMem', 'Voice', 'IR Accuracy')

    # ────────────────────────────────────────────────────────────
    # 1B. WR Accuracy — three 2×2 RM-ANOVAs
    # ────────────────────────────────────────────────────────────
    log("\n── 1B. RM-ANOVAs: WR Accuracy ──")
    if len(cell_wr_valid) > 0:
        _run_two_way_rmanova(pg, cell_wr_valid, 'WR_Rate', 'participant_ID',
                             'SubjectMem', 'ObjectMem', 'WR Accuracy')
        _run_two_way_rmanova(pg, cell_wr_valid, 'WR_Rate', 'participant_ID',
                             'SubjectMem', 'Voice', 'WR Accuracy')
        _run_two_way_rmanova(pg, cell_wr_valid, 'WR_Rate', 'participant_ID',
                             'ObjectMem', 'Voice', 'WR Accuracy')
    else:
        log("  No WR data available. Skipping.")

    # ────────────────────────────────────────────────────────────
    # 1C. RT_IR — three 2×2 RM-ANOVAs
    # ────────────────────────────────────────────────────────────
    log("\n── 1C. RM-ANOVAs: Reaction Time (RT_IR) ──")
    if len(cell_rt_valid) > 0:
        _run_two_way_rmanova(pg, cell_rt_valid, 'MedianRT', 'participant_ID',
                             'SubjectMem', 'ObjectMem', 'RT_IR')
        _run_two_way_rmanova(pg, cell_rt_valid, 'MedianRT', 'participant_ID',
                             'SubjectMem', 'Voice', 'RT_IR')
        _run_two_way_rmanova(pg, cell_rt_valid, 'MedianRT', 'participant_ID',
                             'ObjectMem', 'Voice', 'RT_IR')
    else:
        log("  No RT data available. Skipping.")

    # ────────────────────────────────────────────────────────────
    # 1D. One-Way RM-ANOVA: Block → Hit Rate
    # ────────────────────────────────────────────────────────────
    log("\n── 1D. One-Way RM-ANOVA: Hit Rate ~ Block ──")

    ppb = targets_repeat.groupby(
        ['participant_ID', 'Block']
    )['Accuracy_IR'].mean().reset_index()
    ppb.columns = ['participant_ID', 'Block', 'HitRate']

    blk_counts = ppb.groupby('participant_ID').size()
    valid_blk = blk_counts[blk_counts == 3].index
    ppb_valid = ppb[ppb['participant_ID'].isin(valid_blk)].copy()
    ppb_valid['Block'] = ppb_valid['Block'].astype(str)
    log(f"  Participants with all 3 blocks: {len(valid_blk)}")

    if len(valid_blk) >= 10:
        aov_blk = pg.rm_anova(
            data=ppb_valid, dv='HitRate', subject='participant_ID',
            within='Block', effsize='np2'
        )
        log("\n" + aov_blk.to_string(index=False))

        # Post-hoc pairwise
        posthoc_blk = pg.pairwise_tests(
            data=ppb_valid, dv='HitRate', subject='participant_ID',
            within='Block', padjust='bonf'
        )
        log("\n  Post-hoc (Bonferroni):")
        log(posthoc_blk.to_string(index=False))

    # ── Interaction plots ──
    log("\n  Generating RM-ANOVA interaction plots...")
    sns.set_theme(style="whitegrid", font_scale=1.1)

    cell_counts = cell_ir.groupby('participant_ID').size()
    valid_pids = cell_counts[cell_counts == 8].index
    cell_ir_valid = cell_ir[cell_ir['participant_ID'].isin(valid_pids)].copy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # IR Accuracy interaction
    ax = axes[0]
    if len(valid_pids) >= 10:
        interact_data = cell_ir_valid.groupby(['SubjectMem', 'ObjectMem'])['HitRate'].mean().reset_index()
        interact_wide = interact_data.pivot(index='SubjectMem', columns='ObjectMem', values='HitRate')
        for obj_mem in ['H', 'L']:
            if obj_mem in interact_wide.columns:
                ax.plot(['H', 'L'], [interact_wide.loc['H', obj_mem], interact_wide.loc['L', obj_mem]],
                        marker='o', linewidth=2, markersize=8, label=f'ObjectMem={obj_mem}')
        ax.set_xlabel('Subject Memorability')
        ax.set_ylabel('Mean Hit Rate')
        ax.set_title('IR Accuracy: SubjectMem × ObjectMem', fontweight='bold')
        ax.legend()

    # WR Accuracy interaction
    ax = axes[1]
    if len(cell_wr_valid) > 0 and len(valid_pids_wr) >= 10:
        interact_wr = cell_wr_valid.groupby(['SubjectMem', 'ObjectMem'])['WR_Rate'].mean().reset_index()
        interact_wr_wide = interact_wr.pivot(index='SubjectMem', columns='ObjectMem', values='WR_Rate')
        for obj_mem in ['H', 'L']:
            if obj_mem in interact_wr_wide.columns:
                ax.plot(['H', 'L'], [interact_wr_wide.loc['H', obj_mem], interact_wr_wide.loc['L', obj_mem]],
                        marker='s', linewidth=2, markersize=8, label=f'ObjectMem={obj_mem}')
        ax.set_xlabel('Subject Memorability')
        ax.set_ylabel('Mean WR Accuracy')
        ax.set_title('WR Accuracy: SubjectMem × ObjectMem', fontweight='bold')
        ax.legend()

    # RT interaction
    ax = axes[2]
    if len(valid_pids_rt) >= 10:
        interact_rt = cell_rt_valid.groupby(['SubjectMem', 'ObjectMem'])['MedianRT'].mean().reset_index()
        interact_rt_wide = interact_rt.pivot(index='SubjectMem', columns='ObjectMem', values='MedianRT')
        for obj_mem in ['H', 'L']:
            if obj_mem in interact_rt_wide.columns:
                ax.plot(['H', 'L'], [interact_rt_wide.loc['H', obj_mem], interact_rt_wide.loc['L', obj_mem]],
                        marker='^', linewidth=2, markersize=8, label=f'ObjectMem={obj_mem}')
        ax.set_xlabel('Subject Memorability')
        ax.set_ylabel('Median RT (ms)')
        ax.set_title('RT: SubjectMem × ObjectMem', fontweight='bold')
        ax.legend()

    fig.suptitle('RM-ANOVA Interaction Plots', fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, 'Plot_RMANOVA_Interaction.png')

    # Voice × SubjectMem interaction plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for idx, (dv_label, dv_data, dv_col, valid_set) in enumerate([
        ('Hit Rate', cell_ir_valid, 'HitRate', valid_pids),
        ('WR Accuracy', cell_wr_valid if len(cell_wr_valid) > 0 else pd.DataFrame(), 'WR_Rate', valid_pids_wr),
        ('Median RT', cell_rt_valid, 'MedianRT', valid_pids_rt)
    ]):
        ax = axes[idx]
        if len(dv_data) > 0 and len(valid_set) >= 10:
            interact = dv_data.groupby(['Voice', 'SubjectMem'])[dv_col].mean().reset_index()
            interact_w = interact.pivot(index='Voice', columns='SubjectMem', values=dv_col)
            for sm in ['H', 'L']:
                if sm in interact_w.columns:
                    ax.plot(['A', 'P'], [interact_w.loc['A', sm], interact_w.loc['P', sm]],
                            marker='o', linewidth=2, markersize=8, label=f'SubjectMem={sm}')
            ax.set_xlabel('Voice')
            ax.set_ylabel(dv_label)
            ax.set_title(f'{dv_label}: Voice × SubjectMem', fontweight='bold')
            ax.legend()

    fig.suptitle('Voice × SubjectMem Interaction Plots', fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, 'Plot_RMANOVA_Voice_Interaction.png')



---
## Block 2: Linear Mixed Models (LMMs)

### What are LMMs?
Linear Mixed Models estimate fixed effects (experimental factors) while accounting for the clustering of trials within participants via **random intercepts**. They operate at the individual-trial level rather than aggregated means.

### Why use them here?
- **Complement ANOVA:** RM-ANOVA aggregates to participant-level means, discarding trial-level variability. LMMs preserve every trial, increasing statistical power.
- **Simultaneous modelling:** All predictors (SubjectMem, ObjectMem, Voice, Block) and their interactions are included in one equation.
- **Random effects:** The participant random intercept accounts for individual differences in baseline performance.

### Contrast coding
Predictors are **effect-coded** so coefficients represent the effect of moving between levels:
- SubjectMem: +0.5 = High, −0.5 = Low
- ObjectMem: +0.5 = High, −0.5 = Low  
- Voice: +0.5 = Active, −0.5 = Passive
- Block: centred around the grand mean

### Models fitted
- **2A:** IR Hit (binary) — Linear probability model
- **2B:** WR Accuracy
- **2C:** log(RT_IR) — Log-transformed for normality


### Function: `block2_glmm`

This function fits three LMMs using `statsmodels.formula.api.mixedlm`:

1. **Effect-codes** all predictors and computes interaction terms (SubjObj, SubjVoice, ObjVoice)
2. **Fits LMM for IR_Hit** with participant random intercepts; uses fallback Nelder-Mead optimizer if default convergence fails
3. **Fits LMM for WR accuracy** (only if WR data is available)
4. **Fits LMM for log(RT_IR)** — log-transform normalises the right-skewed RT distribution
5. **Logs a summary** of the contrast coding scheme for interpretability


In [ ]:
def block2_glmm(targets_repeat, hits):
    log("\n" + "=" * 70)
    log("BLOCK 2: GENERALIZED LINEAR MIXED MODELS / LMMs")
    log("=" * 70)

    import statsmodels.formula.api as smf

    # Only WR trials with usable accuracy
    wr_hits = hits.dropna(subset=['Accuracy_WR'])

    # Contrast coding for interpretable fixed effects
    def add_contrasts(df):
        d = df.copy()
        d['SubjectMem_c'] = d['SubjectMem'].map({'H': 0.5, 'L': -0.5})
        d['ObjectMem_c'] = d['ObjectMem'].map({'H': 0.5, 'L': -0.5})
        d['Voice_c'] = d['Voice'].map({'A': 0.5, 'P': -0.5})
        d['Block_c'] = d['Block'] - d['Block'].mean()
        d['SubjObj'] = d['SubjectMem_c'] * d['ObjectMem_c']
        d['SubjVoice'] = d['SubjectMem_c'] * d['Voice_c']
        d['ObjVoice'] = d['ObjectMem_c'] * d['Voice_c']
        return d

    # ────────────────────────────────────────────────────────────
    # 2A. LMM for IR Accuracy (linear probability model)
    # ────────────────────────────────────────────────────────────
    log("\n── 2A. Linear Mixed Model: IR Accuracy (binary) ──")
    log("  Using linear probability model (LPM) with random intercepts")

    tr = add_contrasts(targets_repeat.dropna(subset=['SubjectMem', 'ObjectMem', 'Voice']))
    tr['IR_Hit'] = tr['Accuracy_IR'].astype(float)

    formula_ir = "IR_Hit ~ SubjectMem_c + ObjectMem_c + Voice_c + Block_c + SubjObj + SubjVoice + ObjVoice"

    try:
        model_ir = smf.mixedlm(formula_ir, data=tr, groups=tr['participant_ID'])
        result_ir = model_ir.fit(reml=True)
        log("\n" + str(result_ir.summary()))
    except Exception as e:
        log(f"  LMM for IR failed: {e}")
        # Fallback optimizer if default fails
        try:
            model_ir = smf.mixedlm(formula_ir, data=tr, groups=tr['participant_ID'])
            result_ir = model_ir.fit(reml=True, method='nm')
            log("\n" + str(result_ir.summary()))
        except Exception as e2:
            log(f"  Fallback also failed: {e2}")

    # ────────────────────────────────────────────────────────────
    # 2B. LMM for WR Accuracy
    # ────────────────────────────────────────────────────────────
    log("\n── 2B. Linear Mixed Model: WR Accuracy ──")

    if len(wr_hits) > 0:
        wr = add_contrasts(wr_hits.dropna(subset=['SubjectMem', 'ObjectMem', 'Voice', 'Accuracy_WR']))
        wr['WR'] = wr['Accuracy_WR'].astype(float)
        formula_wr = "WR ~ SubjectMem_c + ObjectMem_c + Voice_c + Block_c + SubjObj + SubjVoice + ObjVoice"

        try:
            model_wr = smf.mixedlm(formula_wr, data=wr, groups=wr['participant_ID'])
            result_wr = model_wr.fit(reml=True)
            log("\n" + str(result_wr.summary()))
        except Exception as e:
            log(f"  LMM for WR failed: {e}")

    # ────────────────────────────────────────────────────────────
    # 2C. LMM for RT_IR (log-transformed)
    # ────────────────────────────────────────────────────────────
    log("\n── 2C. Linear Mixed Model: log(RT_IR) ──")

    rt_data = hits.dropna(subset=['RT_IR', 'SubjectMem', 'ObjectMem', 'Voice']).copy()
    rt_data = rt_data[rt_data['RT_IR'] > 0]

    if len(rt_data) > 100:
        rt = add_contrasts(rt_data)
        rt['logRT'] = np.log(rt['RT_IR'].astype(float))
        formula_rt = "logRT ~ SubjectMem_c + ObjectMem_c + Voice_c + Block_c + SubjObj + SubjVoice + ObjVoice"

        try:
            model_rt = smf.mixedlm(formula_rt, data=rt, groups=rt['participant_ID'])
            result_rt = model_rt.fit(reml=True)
            log("\n" + str(result_rt.summary()))
        except Exception as e:
            log(f"  LMM for RT failed: {e}")

    # ── Summary table ──
    log("\n── GLMM/LMM Summary of Fixed Effects ──")
    log("  (Positive coefficient = higher value for H/Active/later block)")
    log("  SubjectMem_c: +0.5 = High, -0.5 = Low")
    log("  ObjectMem_c:  +0.5 = High, -0.5 = Low")
    log("  Voice_c:      +0.5 = Active, -0.5 = Passive")



---
## Block 3: Regression — Predicting Per-Sentence Memorability

### OLS Regression (sentence-level)
**What it does:** Ordinary Least Squares estimates linear relationships between predictors and a continuous outcome (sentence-level IR/WR memorability scores) by minimising the sum of squared residuals.

**Why use it:** Provides a transparent sentence-level effect map with diagnostic views (coefficient plot, actual-vs-predicted scatter) that visually confirm directionality patterns.

### Logistic Regression (trial-level)
**What it does:** Models the binary IR hit/miss outcome via log-odds. Exponentiating coefficients gives **odds ratios (OR)** — the multiplicative change in hit odds per unit predictor change.

**Why use it:** Statistically appropriate for binary outcomes; provides convergent trial-level evidence expressed as intuitive odds changes.


### Function: `block3_regression`

This function runs four regression models and generates diagnostic plots:

1. **3A — OLS: IR Memorability ~ SubjectMem + ObjectMem + Voice** (392 sentences) — Tests which word properties predict sentence-level recognition
2. **3B — OLS: WR Memorability ~ SubjectMem + ObjectMem + Voice** — Same model for word-recognition scores
3. **3C — Full OLS with interactions + VIF** — Adds SubjObj, SubjVoice, ObjVoice interaction terms; computes Variance Inflation Factors to check multicollinearity
4. **3D — Logistic Regression: IR_Hit ~ Predictors** (5,264 trials) — Binary trial-level model; reports odds ratios with 95% CIs and classification accuracy
5. **Diagnostic plots** — Coefficient bar chart with CIs and actual-vs-predicted scatter


In [ ]:
def block3_regression(targets_repeat, hits, per_sentence_ir, per_sentence_wr):
    log("\n" + "=" * 70)
    log("BLOCK 3: REGRESSION — PREDICTING MEMORABILITY")
    log("=" * 70)

    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    # ── Feature extraction for per-sentence data ──
    ps = per_sentence_ir.copy()
    ps['SubjectMem_num'] = ps['Condition'].str[0].map({'H': 1, 'L': 0})
    ps['ObjectMem_num'] = ps['Condition'].str[1].map({'H': 1, 'L': 0})
    ps['Voice_num'] = ps['Voice'].map({'A': 1, 'P': 0})
    ps['SubjObj_int'] = ps['SubjectMem_num'] * ps['ObjectMem_num']
    ps = ps.dropna(subset=['SubjectMem_num', 'ObjectMem_num', 'Voice_num', 'IR_Memorability'])

    # ────────────────────────────────────────────────────────────
    # 3A. OLS: IR Memorability ~ Word-Level Properties
    # ────────────────────────────────────────────────────────────
    log("\n── 3A. OLS Regression: IR Memorability ~ SubjectMem + ObjectMem + Voice ──")

    formula_3a = "IR_Memorability ~ SubjectMem_num + ObjectMem_num + Voice_num"
    model_3a = smf.ols(formula_3a, data=ps).fit()
    log("\n" + str(model_3a.summary()))

    # ────────────────────────────────────────────────────────────
    # 3B. OLS: WR Memorability ~ Word-Level Properties
    # ────────────────────────────────────────────────────────────
    log("\n── 3B. OLS Regression: WR Memorability ~ SubjectMem + ObjectMem + Voice ──")

    if len(per_sentence_wr) > 0:
        ps_wr = per_sentence_wr.copy()
        ps_wr['SubjectMem_num'] = ps_wr['Condition'].str[0].map({'H': 1, 'L': 0})
        ps_wr['ObjectMem_num'] = ps_wr['Condition'].str[1].map({'H': 1, 'L': 0})
        ps_wr['Voice_num'] = ps_wr['Voice'].map({'A': 1, 'P': 0})
        ps_wr = ps_wr.dropna(subset=['SubjectMem_num', 'ObjectMem_num', 'Voice_num', 'WR_Memorability'])

        formula_3b = "WR_Memorability ~ SubjectMem_num + ObjectMem_num + Voice_num"
        model_3b = smf.ols(formula_3b, data=ps_wr).fit()
        log("\n" + str(model_3b.summary()))

    # ────────────────────────────────────────────────────────────
    # 3C. Full Model with Interactions + VIF
    # ────────────────────────────────────────────────────────────
    log("\n── 3C. Full Regression: IR Memorability ~ All Predictors + Interactions ──")

    ps['SubjVoice_int'] = ps['SubjectMem_num'] * ps['Voice_num']
    ps['ObjVoice_int'] = ps['ObjectMem_num'] * ps['Voice_num']

    formula_3c = ("IR_Memorability ~ SubjectMem_num + ObjectMem_num + Voice_num "
                   "+ SubjObj_int + SubjVoice_int + ObjVoice_int")
    model_3c = smf.ols(formula_3c, data=ps).fit()
    log("\n" + str(model_3c.summary()))

    # VIF for multicollinearity check
    X_vif = ps[['SubjectMem_num', 'ObjectMem_num', 'Voice_num',
                 'SubjObj_int', 'SubjVoice_int', 'ObjVoice_int']].copy()
    X_vif = sm.add_constant(X_vif)
    log("\n  Variance Inflation Factors (VIF):")
    log(f"  {'Variable':<20} {'VIF':>8}")
    log("  " + "-" * 30)
    for i, col in enumerate(X_vif.columns):
        if col == 'const':
            continue
        vif = variance_inflation_factor(X_vif.values, i)
        log(f"  {col:<20} {vif:>8.2f}")

    # ────────────────────────────────────────────────────────────
    # 3D. Logistic Regression at Trial Level
    # ────────────────────────────────────────────────────────────
    log("\n── 3D. Logistic Regression: IR_Hit ~ Predictors (Trial Level) ──")

    tr = targets_repeat.dropna(subset=['SubjectMem', 'ObjectMem', 'Voice']).copy()
    tr['IR_Hit'] = tr['Accuracy_IR'].astype(int)
    tr['SubjectMem_c'] = tr['SubjectMem'].map({'H': 0.5, 'L': -0.5})
    tr['ObjectMem_c'] = tr['ObjectMem'].map({'H': 0.5, 'L': -0.5})
    tr['Voice_c'] = tr['Voice'].map({'A': 0.5, 'P': -0.5})
    tr['Block_c'] = tr['Block'].astype(float) - tr['Block'].astype(float).mean()
    tr['SubjObj'] = tr['SubjectMem_c'] * tr['ObjectMem_c']

    formula_logit = "IR_Hit ~ SubjectMem_c + ObjectMem_c + Voice_c + Block_c + SubjObj"
    try:
        logit_model = smf.logit(formula_logit, data=tr).fit(disp=0)
        log("\n" + str(logit_model.summary()))

        # Odds ratios
        log("\n  Odds Ratios (exp(β)):")
        log(f"  {'Predictor':<20} {'OR':>8} {'95% CI Lower':>14} {'95% CI Upper':>14}")
        log("  " + "-" * 60)
        params = logit_model.params
        conf = logit_model.conf_int()
        for var in params.index:
            if var == 'Intercept':
                continue
            or_val = np.exp(params[var])
            ci_lo = np.exp(conf.loc[var, 0])
            ci_hi = np.exp(conf.loc[var, 1])
            log(f"  {var:<20} {or_val:>8.4f} {ci_lo:>14.4f} {ci_hi:>14.4f}")

        # Classification accuracy
        pred_prob = logit_model.predict(tr)
        pred_class = (pred_prob >= 0.5).astype(int)
        accuracy = (pred_class == tr['IR_Hit']).mean()
        log(f"\n  Classification Accuracy: {accuracy:.4f}")
        log(f"  Baseline (always predict majority class): {tr['IR_Hit'].mean():.4f}")

    except Exception as e:
        log(f"  Logistic regression failed: {e}")

    # ── Regression diagnostic plots ──
    log("\n  Generating regression plots...")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Coefficient plot for OLS (3A)
    ax = axes[0]
    coefs = model_3a.params.drop('Intercept')
    ci = model_3a.conf_int().drop('Intercept')
    y_pos = range(len(coefs))
    ax.barh(y_pos, coefs.values, xerr=[coefs.values - ci[0].values, ci[1].values - coefs.values],
            color=['#2ecc71' if c > 0 else '#e74c3c' for c in coefs.values],
            edgecolor='black', alpha=0.8, capsize=5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(coefs.index)
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Coefficient (β)')
    ax.set_title('OLS Coefficients: IR Memorability', fontweight='bold')

    # Actual vs predicted scatter (3A)
    ax = axes[1]
    predicted = model_3a.predict(ps)
    ax.scatter(predicted, ps['IR_Memorability'], alpha=0.3, s=20, color='#2c3e50')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Predicted IR Memorability')
    ax.set_ylabel('Actual IR Memorability')
    ax.set_title('Actual vs Predicted (OLS)', fontweight='bold')
    ax.legend()

    fig.suptitle('Regression Diagnostics', fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, 'Plot_Regression_Diagnostics.png')



---
## Block 4: Signal Detection Analysis (d′ and Criterion c)

### What is Signal Detection Theory?
Signal Detection Theory (SDT) separates **true recognition sensitivity** from **response bias**. Raw hit rates alone can be misleading — a participant who always says "yes" has a high hit rate but also high false alarms.

### Key formulae
- **Sensitivity:** d′ = Z(Hit Rate) − Z(False Alarm Rate)  
  Higher d′ = better old/new discrimination, independent of bias.
- **Criterion c:** c = −½ · [Z(Hit Rate) + Z(False Alarm Rate)]  
  Positive c = conservative (tends to say "no"); negative c = liberal.

### Floor/ceiling correction
When HR = 1.0 or FAR = 0.0, the z-transform becomes infinite. The standard correction bounds rates using 1/(2N) and 1−1/(2N).

### Why essential here?
Confirms that condition differences reflect genuine sensitivity changes, not just shifts in willingness to respond "old."


### Function: `block4_dprime`

This function computes signal detection metrics and tests them statistically:

1. **4A — Compute d′ per participant × condition:** Merges hit rates with participant-level false-alarm rates, applies floor/ceiling correction, then computes d′ and criterion c
2. **4B — RM-ANOVA on d′ by condition:** Tests whether sensitivity differs across HH/HL/LH/LL; runs Bonferroni-corrected post-hoc comparisons
3. **4C — RM-ANOVA on criterion c:** Tests whether response bias shifts across conditions
4. **4D — d′ by Voice:** Paired t-test comparing active vs. passive voice sensitivity
5. **Figures:** Bar charts for d′ and criterion c by condition, d′ by voice, and violin plots showing full d′ distributions


In [ ]:
def block4_dprime(targets_repeat, fa_rate_df):
    log("\n" + "=" * 70)
    log("BLOCK 4: SIGNAL DETECTION ANALYSIS (d′)")
    log("=" * 70)

    try:
        import pingouin as pg
    except ImportError:
        log("  ERROR: pingouin not installed for RM-ANOVA on d'. Run: pip install pingouin")
        pg = None

    conds = ['HH', 'HL', 'LH', 'LL']

    # ────────────────────────────────────────────────────────────
    # 4A. Compute d′ per Participant × Condition
    # ────────────────────────────────────────────────────────────
    log("\n── 4A. d′ per Participant × Condition ──")

    # Per-participant per-condition hit rate
    ppc_hr = targets_repeat.groupby(['participant_ID', 'Condition']).agg(
        n_hits=('Accuracy_IR', 'sum'),
        n_trials=('Accuracy_IR', 'count')
    ).reset_index()
    ppc_hr['HR'] = ppc_hr['n_hits'] / ppc_hr['n_trials']

    # Merge FA rate (per-participant, pooled across conditions)
    ppc_hr = ppc_hr.merge(fa_rate_df[['participant_ID', 'FA_Rate']], on='participant_ID', how='left')
    ppc_hr['FA_Rate'] = ppc_hr['FA_Rate'].fillna(0)

    # Floor/ceiling correction
    def correct_rate(rate, n):
        floor = 1.0 / (2.0 * n)
        ceiling = 1.0 - 1.0 / (2.0 * n)
        return np.clip(rate, floor, ceiling)

    ppc_hr['HR_corr'] = ppc_hr.apply(lambda r: correct_rate(r['HR'], max(r['n_trials'], 1)), axis=1)

    # For FA rate, use a reasonable N (total lures per participant, approximate)
    # Use n_lure from fa_rate_df
    fa_with_n = fa_rate_df.copy()
    if 'n_lure' in fa_with_n.columns:
        ppc_hr = ppc_hr.merge(fa_with_n[['participant_ID', 'n_lure']], on='participant_ID', how='left')
        ppc_hr['n_lure'] = ppc_hr['n_lure'].fillna(170)  # approximate default
    else:
        ppc_hr['n_lure'] = 170  # approximate

    ppc_hr['FAR_corr'] = ppc_hr.apply(lambda r: correct_rate(r['FA_Rate'], max(r['n_lure'], 1)), axis=1)

    # Compute d′ and c
    ppc_hr['dprime'] = norm.ppf(ppc_hr['HR_corr']) - norm.ppf(ppc_hr['FAR_corr'])
    ppc_hr['criterion_c'] = -0.5 * (norm.ppf(ppc_hr['HR_corr']) + norm.ppf(ppc_hr['FAR_corr']))

    # Descriptive stats
    log(f"\n  {'Condition':<10} {'N':>5} {'Mean d′':>10} {'SD d′':>10} {'95% CI Lower':>14} {'95% CI Upper':>14}")
    log("  " + "-" * 70)
    from scipy.stats import t as t_dist
    for c in conds:
        vals = ppc_hr[ppc_hr['Condition'] == c]['dprime'].dropna()
        n = len(vals)
        if n >= 2:
            mean = vals.mean()
            sd = vals.std(ddof=1)
            se = sd / np.sqrt(n)
            t_crit = t_dist.ppf(0.975, df=n - 1)
            ci_lo = mean - t_crit * se
            ci_hi = mean + t_crit * se
            log(f"  {c:<10} {n:>5} {mean:>10.4f} {sd:>10.4f} {ci_lo:>14.4f} {ci_hi:>14.4f}")

    log(f"\n  Overall mean d′: {ppc_hr['dprime'].mean():.4f} (SD={ppc_hr['dprime'].std():.4f})")

    # Criterion c descriptives
    log(f"\n  {'Condition':<10} {'Mean c':>10} {'SD c':>10}")
    log("  " + "-" * 35)
    for c in conds:
        vals = ppc_hr[ppc_hr['Condition'] == c]['criterion_c'].dropna()
        if len(vals) >= 2:
            log(f"  {c:<10} {vals.mean():>10.4f} {vals.std():>10.4f}")

    # ────────────────────────────────────────────────────────────
    # 4B. RM-ANOVA on d′ by Condition
    # ────────────────────────────────────────────────────────────
    log("\n── 4B. RM-ANOVA: d′ ~ Condition ──")

    dprime_wide = ppc_hr.pivot_table(index='participant_ID', columns='Condition',
                                      values='dprime', aggfunc='first')
    valid_dp = dprime_wide.dropna()
    log(f"  Participants with d′ for all 4 conditions: {len(valid_dp)}")

    if pg and len(valid_dp) >= 10:
        dprime_long = valid_dp.reset_index().melt(
            id_vars='participant_ID', value_vars=conds,
            var_name='Condition', value_name='dprime'
        )
        aov_dp = pg.rm_anova(
            data=dprime_long, dv='dprime', subject='participant_ID',
            within='Condition', effsize='np2'
        )
        log("\n" + aov_dp.to_string(index=False))

        # Post-hoc
        posthoc_dp = pg.pairwise_tests(
            data=dprime_long, dv='dprime', subject='participant_ID',
            within='Condition', padjust='bonf'
        )
        log("\n  Post-hoc (Bonferroni):")
        log(posthoc_dp.to_string(index=False))
    else:
        # Fallback: Kruskal-Wallis
        log("  Falling back to KW test...")
        groups = [valid_dp[c].values for c in conds if c in valid_dp.columns]
        if len(groups) >= 2:
            h, p = stats.kruskal(*groups)
            log(f"  KW H = {h:.4f}, p = {p:.4e}")

    # ────────────────────────────────────────────────────────────
    # 4C. RM-ANOVA on Criterion c
    # ────────────────────────────────────────────────────────────
    log("\n── 4C. RM-ANOVA: Criterion c ~ Condition ──")

    c_wide = ppc_hr.pivot_table(index='participant_ID', columns='Condition',
                                 values='criterion_c', aggfunc='first')
    valid_c = c_wide.dropna()
    log(f"  Participants with c for all 4 conditions: {len(valid_c)}")

    if pg and len(valid_c) >= 10:
        c_long = valid_c.reset_index().melt(
            id_vars='participant_ID', value_vars=conds,
            var_name='Condition', value_name='criterion_c'
        )
        aov_c = pg.rm_anova(
            data=c_long, dv='criterion_c', subject='participant_ID',
            within='Condition', effsize='np2'
        )
        log("\n" + aov_c.to_string(index=False))

    # ────────────────────────────────────────────────────────────
    # 4D. d′ by Voice
    # ────────────────────────────────────────────────────────────
    log("\n── 4D. d′ by Voice (Active vs Passive) ──")

    ppv_hr = targets_repeat.groupby(['participant_ID', 'Voice']).agg(
        n_hits=('Accuracy_IR', 'sum'),
        n_trials=('Accuracy_IR', 'count')
    ).reset_index()
    ppv_hr['HR'] = ppv_hr['n_hits'] / ppv_hr['n_trials']
    ppv_hr = ppv_hr.merge(fa_rate_df[['participant_ID', 'FA_Rate']], on='participant_ID', how='left')
    ppv_hr['FA_Rate'] = ppv_hr['FA_Rate'].fillna(0)

    if 'n_lure' in fa_with_n.columns:
        ppv_hr = ppv_hr.merge(fa_with_n[['participant_ID', 'n_lure']], on='participant_ID', how='left')
        ppv_hr['n_lure'] = ppv_hr['n_lure'].fillna(170)
    else:
        ppv_hr['n_lure'] = 170

    ppv_hr['HR_corr'] = ppv_hr.apply(lambda r: correct_rate(r['HR'], max(r['n_trials'], 1)), axis=1)
    ppv_hr['FAR_corr'] = ppv_hr.apply(lambda r: correct_rate(r['FA_Rate'], max(r['n_lure'], 1)), axis=1)
    ppv_hr['dprime'] = norm.ppf(ppv_hr['HR_corr']) - norm.ppf(ppv_hr['FAR_corr'])

    dprime_voice_wide = ppv_hr.pivot_table(index='participant_ID', columns='Voice',
                                            values='dprime', aggfunc='first').dropna()

    if 'A' in dprime_voice_wide.columns and 'P' in dprime_voice_wide.columns:
        mean_a = dprime_voice_wide['A'].mean()
        mean_p = dprime_voice_wide['P'].mean()
        log(f"  d′ Active:  M={mean_a:.4f}, SD={dprime_voice_wide['A'].std():.4f}")
        log(f"  d′ Passive: M={mean_p:.4f}, SD={dprime_voice_wide['P'].std():.4f}")

        t_stat, p_val = stats.ttest_rel(dprime_voice_wide['A'], dprime_voice_wide['P'])
        n = len(dprime_voice_wide)
        d = t_stat / np.sqrt(n)
        log(f"  Paired t-test: t({n-1}) = {t_stat:.4f}, p = {p_val:.4e}, d = {d:.4f}")
        log(f"  {'*** SIGNIFICANT ***' if p_val < 0.05 else 'Not significant'}")

    # ── d′ Figures ──
    log("\n  Generating d′ plots...")
    palette = {'HH': '#2ecc71', 'HL': '#3498db', 'LH': '#e67e22', 'LL': '#e74c3c'}
    voice_pal = {'A': '#3498db', 'P': '#e74c3c'}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # d′ bar chart by condition
    ax = axes[0]
    dp_means = ppc_hr.groupby('Condition')['dprime'].agg(['mean', 'std', 'count'])
    dp_means['se'] = dp_means['std'] / np.sqrt(dp_means['count'])
    bars = ax.bar(conds, [dp_means.loc[c, 'mean'] for c in conds],
                  yerr=[dp_means.loc[c, 'se'] for c in conds],
                  color=[palette[c] for c in conds], edgecolor='black',
                  capsize=5, alpha=0.85)
    ax.set_xlabel('Condition')
    ax.set_ylabel("d′")
    ax.set_title("d′ by Condition", fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5)

    # Criterion c bar chart
    ax = axes[1]
    c_means = ppc_hr.groupby('Condition')['criterion_c'].agg(['mean', 'std', 'count'])
    c_means['se'] = c_means['std'] / np.sqrt(c_means['count'])
    bars = ax.bar(conds, [c_means.loc[c, 'mean'] for c in conds],
                  yerr=[c_means.loc[c, 'se'] for c in conds],
                  color=[palette[c] for c in conds], edgecolor='black',
                  capsize=5, alpha=0.85)
    ax.set_xlabel('Condition')
    ax.set_ylabel("Criterion c")
    ax.set_title("Response Criterion by Condition", fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5)

    # d′ by voice
    ax = axes[2]
    if len(dprime_voice_wide) > 0:
        voice_means = ppv_hr.groupby('Voice')['dprime'].agg(['mean', 'std', 'count'])
        voice_means['se'] = voice_means['std'] / np.sqrt(voice_means['count'])
        for i, v in enumerate(['A', 'P']):
            if v in voice_means.index:
                ax.bar(i, voice_means.loc[v, 'mean'], yerr=voice_means.loc[v, 'se'],
                       color=voice_pal[v], edgecolor='black', capsize=5, alpha=0.85,
                       label=f"{'Active' if v == 'A' else 'Passive'}")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Active', 'Passive'])
        ax.set_ylabel("d′")
        ax.set_title("d′ by Voice", fontweight='bold')
        ax.legend()
        ax.axhline(0, color='black', linewidth=0.5)

    fig.suptitle("Signal Detection Analysis", fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, 'Plot_dprime_Analysis.png')

    # d′ violin by condition
    fig, ax = plt.subplots(figsize=(9, 6))
    dprime_long_all = ppc_hr[ppc_hr['Condition'].isin(conds)][['participant_ID', 'Condition', 'dprime']]
    sns.violinplot(data=dprime_long_all, x='Condition', y='dprime', order=conds,
                   palette=palette, inner='quartile', cut=0, ax=ax)
    ax.set_title("d′ Distribution by Condition — Violin", fontweight='bold')
    ax.set_ylabel("d′")
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    save_fig(fig, 'Plot_dprime_Violin.png')



---
## Run All Analysis Blocks

Execute all four blocks sequentially and save the complete results log to disk.


In [ ]:
log("=" * 70)
log("  SENTENCE MEMORABILITY — PHASE 2: ADVANCED ANALYSIS")
log("=" * 70)

# Run all blocks
block1_rmanova(targets_repeat, hits)
block2_glmm(targets_repeat, hits)
block3_regression(targets_repeat, hits, per_sentence_ir, per_sentence_wr)
block4_dprime(targets_repeat, fa_rate_df)

save_results()
log("\nDone — Phase 2 analysis complete.")
